### Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from pathlib import Path
import sys
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ All imports successful")

✅ All imports successful


### Database Connection 

In [2]:
# Define database path
db_path = Path(r'D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\data\ETH.db')

if not db_path.exists():
    print(f"❌ Database not found at: {db_path}")
    # Try relative path
    db_path = Path('../../../../data/ETH.db').resolve()
    if not db_path.exists():
        print("❌ Database still not found!")
        print(f"Current directory: {Path.cwd()}")
        raise FileNotFoundError("ETH.db not found")

print(f"✅ Database found: {db_path}")

# Connect to database
conn = sqlite3.connect(str(db_path))
print("✅ Connected to database")

✅ Database found: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\data\ETH.db
✅ Connected to database


### List Tables

In [3]:
# Get all tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("=" * 50)
print("📋 TABLES IN DATABASE")
print("=" * 50)

if tables.empty:
    print("❌ No tables found!")
else:
    for _, row in tables.iterrows():
        table_name = row['name']
        # Get row count
        count = pd.read_sql(f"SELECT COUNT(*) as count FROM {table_name};", conn).iloc[0, 0]
        print(f"  📊 {table_name:20s} : {count:>8,} rows")

# Check if eth_ohlcv exists
if 'eth_ohlcv' not in tables['name'].values:
    print("\n❌ Table 'eth_ohlcv' not found!")
    print("Available tables:", tables['name'].tolist())
    conn.close()
    raise ValueError("eth_ohlcv table not in database")

📋 TABLES IN DATABASE
  📊 eth_daily            :      365 rows
  📊 eth_signals          :      365 rows
  📊 trades               :        5 rows
  📊 sqlite_sequence      :        2 rows
  📊 performance          :        0 rows
  📊 signals              :       15 rows
  📊 eth_ohlcv            :      395 rows


### Load Data with Column Check

In [4]:
# First, check column names
sample = pd.read_sql("SELECT * FROM eth_ohlcv LIMIT 1;", conn)
print("📋 Available columns:")
print(sample.columns.tolist())

# Load all data
df = pd.read_sql("SELECT * FROM eth_ohlcv ORDER BY date;", conn)

# Check if 'date' column exists (might be 'Date' or 'timestamp')
date_col = None
for col in ['date', 'Date', 'timestamp', 'Timestamp', 'time']:
    if col in df.columns:
        date_col = col
        break

if date_col:
    df[date_col] = pd.to_datetime(df[date_col])
    df.set_index(date_col, inplace=True)
else:
    print("⚠️  No date column found, using integer index")

# Check for price columns
price_col = None
for col in ['close', 'Close', 'price', 'Price']:
    if col in df.columns:
        price_col = col
        break

if price_col is None:
    print("❌ No price column found!")
    print("Available columns:", df.columns.tolist())
    conn.close()
    raise ValueError("No price column found")

print(f"\n✅ Using price column: '{price_col}'")
print(f"✅ Loaded {len(df):,} rows")
print(f"📅 Date range: {df.index.min()} to {df.index.max()}" if date_col else "📅 No date column")

conn.close()

# Show sample
display(df.head())

📋 Available columns:
['date', 'open', 'high', 'low', 'close', 'volume', 'SMA_7', 'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26', 'MACD', 'MACD_signal', 'MACD_histogram', 'RSI_14', 'BB_middle', 'BB_upper', 'BB_lower', 'ATR_14']

✅ Using price column: 'close'
✅ Loaded 395 rows
📅 Date range: 2025-07-29 00:00:00 to 2026-07-28 00:00:00


,open,high,low,close,volume,SMA_7,SMA_20,SMA_50,EMA_12,EMA_26,MACD,MACD_signal,MACD_histogram,RSI_14,BB_middle,BB_upper,BB_lower,ATR_14
date,,,,,,,,,,,,,,,,,,
2025-07-29,3799.00,3886.44,3716.04,3793.79,584210.4393,NaN,NaN,NaN,3793.790000,3793.790000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
2025-07-30,3793.79,3834.03,3677.65,3810.00,525424.9003,NaN,NaN,NaN,3796.283846,3794.990741,1.293105,0.258621,1.034484,NaN,NaN,NaN,NaN,NaN
2025-07-31,3810.00,3878.67,3684.33,3698.39,516088.4503,NaN,NaN,NaN,3781.223254,3787.835130,-6.611876,-1.115478,-5.496398,NaN,NaN,NaN,NaN,NaN
2025-08-01,3698.39,3724.02,3431.75,3488.20,783869.1517,NaN,NaN,NaN,3736.142754,3765.639935,-29.497182,-6.791819,-22.705363,NaN,NaN,NaN,NaN,NaN
2025-08-02,3488.21,3537.69,3368.29,3393.94,494195.7094,NaN,NaN,NaN,3683.496176,3738.106607,-54.610431,-16.355541,-38.254889,NaN,NaN,NaN,NaN,NaN
